# Signed Jumps 04b — real-time seconds trigger

Protocol: `discussions/Signed_Jumps/009_claude_sj04b_preregistration.md` (fixed before this run).

Trigger at the first second where the trailing-60 s return exceeds c × the SJ 01 engine's 1-minute σ (past data only), then measure the signed return from trigger + δ to trigger + δ + H. Gate: c = 4, δ = 2 s, H = 60 s, gross q2.5 > 0.

In [1]:
import os,sys,json,time,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
from dataclasses import replace
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import signed_jumps as sj, signed_jumps_seconds as ss, signed_jumps_trigger as st
for m in (sj,ss,st):importlib.reload(m)
assert st.VERSION=='SJ04b-0.1.0'
SYNTHETIC=os.getenv('SJ04B_SYNTHETIC','0')=='1'
p=sj.Params();tp=st.TrigParams()
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='sj04b-');OUT=Path(tmp.name)/'05'
    p=sj.Params(n_boot=200,half_lives=(1,));tp=st.TrigParams(n_boot=300)
else:
    OUT=ROOT/'notebooks'/'Signed_Jumps'/'05'
OUT.mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
if not SYNTHETIC:
    chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_signed_jumps_trigger.py')],cwd=str(ROOT),env=env,capture_output=True,text=True)
    print(chk.stdout[-600:]);print(chk.stderr[-400:]);assert chk.returncode==0,'tests failed'
print(json.dumps(st.registration(tp),indent=1))

.....                                                                    [100%]
5 passed in 6.45s


{
 "c_trig": 4.0,
 "c_grid": [
  3.0,
  4.0,
  5.0
 ],
 "floor_bp": 5.0,
 "window_s": 60,
 "cooldown_s": 300,
 "delta_s": 2,
 "h_s": 60,
 "grid_delta": [
  1,
  2,
  5,
  10,
  30
 ],
 "grid_h": [
  10,
  30,
  60,
  300
 ],
 "costs_bp": [
  7.0,
  12.0
 ],
 "n_boot": 2000,
 "status": "PREREGISTERED_SJ009",
 "version": "SJ04b-0.1.0",
 "gate": "c_trig=4, delta=2 s, H=60 s: gross day-bootstrap q2.5 > 0 => CONTINUATION; else NO_CONTINUATION"
}


## Engine σ (SJ 01, unchanged) and 1-second data

In [2]:
T0=time.time()
if SYNTHETIC:
    secs,one=ss.simulate_seconds(n_days=14,n_assets=len(p.assets),seed=4,drift_bp=6,drift_s=60)
    secs={a:v for a,v in zip(p.assets,secs.values())};one={a:v for a,v in zip(p.assets,one.values())}
    assets,pan,_=sj.prepare(one,p)
else:
    BIN=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent/'Binance'
    S={A:sj.load_spot_1m(BIN/'basis'/'spot'/f'{A}USDT_spot_1m.parquet') for A in p.assets}
    assets,pan,_=sj.prepare(S,p)
    secs={A:ss.load_spot_1s(sorted((BIN/'spot_1s'/f'{A}USDT').glob(f'{A}USDT_spot_1s_*.parquet'))) for A in p.assets}
sig={a.name:st.sigma_minute(a) for a in assets}
for a in assets:print(a.name,'median 1-min sigma (bp):',round(float(sig[a.name].loc[secs[a.name].index.min():secs[a.name].index.max()].median()*1e4),2))
print(f'{time.time()-T0:.0f}s')

D:\Projects\deribit_market_data\src\data_engineering\modeling\signed_jumps.py:172: RuntimeWarning: divide by zero encountered in log
  feats.append(np.log(m))
D:\Projects\deribit_market_data\src\data_engineering\modeling\signed_jumps.py:187: RuntimeWarning: divide by zero encountered in divide
  Z = z / np.sqrt(sig2c)
D:\Projects\deribit_market_data\src\data_engineering\modeling\signed_jumps.py:187: RuntimeWarning: invalid value encountered in divide
  Z = z / np.sqrt(sig2c)


BTC median 1-min sigma (bp): 3.99
ETH median 1-min sigma (bp): 5.49
SOL median 1-min sigma (bp): 6.37
XRP median 1-min sigma (bp): 6.13
138s


## Triggers, forward returns, grid

In [3]:
tabs=[];evs={}
for c in tp.c_grid:
    parts=[]
    for a in assets:
        tr=st.triggers(secs[a.name],sig[a.name],c,tp)
        f=st.forward(tr,secs[a.name],tp);f['asset']=a.name;parts.append(f)
    ev=pd.concat(parts,ignore_index=True);evs[c]=ev
    tabs.append(st.grid_table(ev,tp,c))
tab=pd.concat(tabs,ignore_index=True);tab.to_csv(OUT/'05_grid.csv',index=False)
evs[tp.c_trig].to_csv(OUT/'05_triggers_c4.csv.gz',index=False)
print('triggers per c:',{c:len(e) for c,e in evs.items()})
for c in tp.c_grid:
    print(f'c = {c}: gross bp (rows = latency delta s, cols = hold H s)')
    display(tab[tab.c_trig==c].pivot_table(index='delta_s',columns='h_s',values='gross_bp').round(2))
print('c = 4: 95% lower bounds');display(tab[tab.c_trig==tp.c_trig].pivot_table(index='delta_s',columns='h_s',values='lo').round(2))
print('c = 4: share positive');display(tab[tab.c_trig==tp.c_trig].pivot_table(index='delta_s',columns='h_s',values='share_positive').round(3))

triggers per c: {3.0: 13001, 4.0: 4913, 5.0: 2296}
c = 3.0: gross bp (rows = latency delta s, cols = hold H s)


h_s,10,30,60,300
delta_s,,,,
1,0.17,0.03,-0.08,-0.26
2,0.16,0.03,-0.03,-0.23
5,0.00,-0.14,-0.22,-0.37
10,-0.14,-0.24,-0.27,-0.45
30,-0.08,-0.08,-0.25,-0.21


c = 4.0: gross bp (rows = latency delta s, cols = hold H s)


h_s,10,30,60,300
delta_s,,,,
1,0.19,0.04,0.23,0.00
2,0.15,0.06,0.25,0.02
5,0.02,-0.12,0.03,-0.10
10,-0.22,-0.21,-0.22,-0.25
30,0.02,0.18,-0.24,0.06


c = 5.0: gross bp (rows = latency delta s, cols = hold H s)


h_s,10,30,60,300
delta_s,,,,
1,0.29,0.28,0.32,0.34
2,0.34,0.35,0.40,0.50
5,0.02,0.13,0.14,0.31
10,0.06,0.04,-0.14,0.36
30,-0.05,0.02,-0.71,0.29


c = 4: 95% lower bounds


h_s,10,30,60,300
delta_s,,,,
1,-0.15,-0.49,-0.48,-1.43
2,-0.20,-0.50,-0.46,-1.27
5,-0.32,-0.68,-0.58,-1.45
10,-0.43,-0.69,-0.82,-1.58
30,-0.21,-0.18,-0.76,-1.26


c = 4: share positive


h_s,10,30,60,300
delta_s,,,,
1,0.439,0.440,0.438,0.429
2,0.435,0.441,0.447,0.433
5,0.432,0.445,0.444,0.427
10,0.413,0.431,0.431,0.426
30,0.421,0.441,0.432,0.441


## Splits (c = 4, δ = 2 s, H = 60 s)

In [4]:
e=evs[tp.c_trig];k=f'g_{tp.delta_s}_{tp.h_s}'
day=pd.DatetimeIndex(e['tau']).normalize().asi8
rows=[]
for name,m in [('all',np.ones(len(e),bool))]+[(f'asset={a}',(e.asset==a).to_numpy()) for a in p.assets]+[('dir=up',(e.dir>0).to_numpy()),('dir=down',(e.dir<0).to_numpy())]:
    mm,lo,hi,n=st.boot_mean(e[k].to_numpy()[m],day[m],tp.n_boot,5)
    rows.append({'split':name,'gross_bp':mm,'lo':lo,'hi':hi,'n':n})
spl=pd.DataFrame(rows);display(spl.round(2));spl.to_csv(OUT/'05_splits.csv',index=False)
print('trigger z / R60 (bp) quantiles:');display(e[['z','R60_bp']].abs().quantile([.1,.5,.9]).round(2))

,split,gross_bp,lo,hi,n
0,all,0.25,-0.45,1.05,4913
1,asset=BTC,0.25,-0.41,0.91,1289
2,asset=ETH,0.59,-0.41,1.76,1242
3,asset=SOL,-0.44,-1.40,0.56,1188
4,asset=XRP,0.57,-0.63,1.85,1194
5,dir=up,0.29,-0.79,1.41,2516
6,dir=down,0.21,-0.62,1.19,2397


trigger z / R60 (bp) quantiles:


,z,R60_bp
0.1,4.03,10.15
0.5,4.21,22.93
0.9,5.18,46.34


## Decision

In [5]:
dec=st.decide(tab,tp);dec.update(triggers={str(c):int(len(v)) for c,v in evs.items()},synthetic=SYNTHETIC)
print(json.dumps(dec,indent=1,default=float))
if not SYNTHETIC:
    (OUT/'05_decision.json').write_text(json.dumps(dec,indent=1,default=float))
    (OUT/'05_manifest.json').write_text(json.dumps({'registration':st.registration(tp),'runtime_s':time.time()-T0},indent=1,default=str))
print(f'total {time.time()-T0:.0f}s')

{
 "version": "SJ04b-0.1.0",
 "gross_bp": 0.24954636266250896,
 "q025": -0.4623568795612529,
 "q975": 1.0312781843863394,
 "n": 4913,
 "verdict": "NO_CONTINUATION",
 "gross_ge_7bp": false,
 "net_7bp": -6.7504536373374915,
 "gross_ge_12bp": false,
 "net_12bp": -11.750453637337491,
 "triggers": {
  "3.0": 13001,
  "4.0": 4913,
  "5.0": 2296
 },
 "synthetic": false
}
total 150s
